# BERTweet Fine-tuning — Mental Health Sentiment Analysis

End-to-end notebook:
1. Environment check
2. Config & path setup
3. Preprocessing (if needed)
4. Data loaders
5. Model, optimiser & scheduler
6. Training loop
7. Training curves
8. Evaluation on test split


## 1 — Environment check

In [ ]:
import subprocess, sys, importlib

REQUIRED = ['torch', 'transformers', 'sklearn', 'tqdm', 'yaml', 'pandas', 'matplotlib', 'seaborn']
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}  '
      f'({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"})')


## 2 — Config & path setup

Edit the commented-out overrides to change settings without touching the YAML.

In [ ]:
import sys, os
from pathlib import Path

# Notebook should sit at the project root; adjust if you moved it
PROJECT_ROOT = Path('/home/sakana/Code/PTIT/NLP/sentimind')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('Working dir:', PROJECT_ROOT)


In [ ]:
import yaml, json

with open('configs/bertweet.yaml') as f:
    cfg = yaml.safe_load(f)

#  Optional overrides (uncomment to activate) 
# cfg['training']['epochs'] = 3          # quick smoke-test
# cfg['training']['fp16'] = True          # enable if CUDA GPU available
# cfg['model']['freeze_base'] = True      # head-only training (faster)
# 

print(json.dumps(cfg, indent=2))


In [ ]:
import torch
from src.training.trainer import set_seed

SEED = cfg['seed']
set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', DEVICE)

data_cfg  = cfg['data']
model_cfg = cfg['model']
train_cfg = cfg['training']
out_cfg   = cfg['output']

ARTIFACTS_DIR = Path(out_cfg['artifacts_dir'])
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


## 3 — Preprocessing (if needed)

Runs `scripts/preprocess.py` only when `data/processed/*.csv` is missing.

In [ ]:
train_csv = Path(data_cfg['train_path'])

if not train_csv.exists():
    print('Processed data not found — running preprocess.py ...')
    result = subprocess.run(
        [sys.executable, 'scripts/preprocess.py', '--config', 'configs/preprocessing.yaml'],
        capture_output=True, text=True,
    )
    print(result.stdout[-3000:] if result.stdout else '')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:])
        raise RuntimeError('Preprocessing failed — see output above')
else:
    print(f'Processed data already at {train_csv.parent}')

import pandas as pd
for split, path in [('train', data_cfg['train_path']),
                    ('val',   data_cfg['val_path']),
                    ('test',  data_cfg['test_path'])]:
    df = pd.read_csv(path)
    print(f'  {split:5s}: {len(df):6,} rows  |  columns: {list(df.columns)}')


## 4 — Data loaders

In [ ]:
from src.data.bertweet_dataset import build_transformer_loaders

train_loader, val_loader, test_loader = build_transformer_loaders(
    train_path=data_cfg['train_path'],
    val_path=data_cfg['val_path'],
    test_path=data_cfg['test_path'],
    model_name=model_cfg['pretrained_name'],
    max_len=data_cfg['max_seq_len'],
    batch_size=train_cfg['batch_size'],
    seed=SEED,
)

print(f'Batches — train: {len(train_loader):,}  val: {len(val_loader):,}  test: {len(test_loader):,}')
sample = next(iter(train_loader))
print('Batch keys :', list(sample.keys()))
print('input_ids  :', sample['input_ids'].shape)
print('labels     :', sample['labels'][:8].tolist())


In [ ]:
import numpy as np

num_classes   = model_cfg['num_classes']
class_weights = None

if train_cfg.get('class_weighted_loss', True):
    df_train = pd.read_csv(data_cfg['train_path'])
    counts = np.zeros(num_classes)
    for lbl, cnt in df_train['label_id'].value_counts().items():
        counts[int(lbl)] = cnt
    counts = np.where(counts == 0, 1, counts)
    weights = 1.0 / counts
    weights = weights / weights.sum()
    class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    from src.data.preprocess import ID_TO_LABEL
    print('Class weights:')
    for i, w in enumerate(weights):
        print(f'  [{i}] {ID_TO_LABEL.get(i, "?"):25s} count={int(counts[i]):6,}  weight={w:.4f}')


## 5 — Model, optimiser & scheduler

In [ ]:
from src.models.bertweet import BERTweetClassifier

model = BERTweetClassifier(
    model_name=model_cfg['pretrained_name'],
    num_classes=num_classes,
    dropout=model_cfg['dropout'],
    freeze_base=model_cfg.get('freeze_base', False),
)
model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parameters: {trainable:,} trainable / {total:,} total  ({trainable/total*100:.1f}%)')


In [ ]:
import torch.nn as nn
from transformers import get_linear_schedule_with_warmup
from src.training.trainer import EarlyStopping

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_cfg['learning_rate'],
    weight_decay=train_cfg['weight_decay'],
)

ACCUM_STEPS = train_cfg.get('gradient_accumulation_steps', 1)
EPOCHS      = train_cfg['epochs']

total_steps  = (len(train_loader) // ACCUM_STEPS) * EPOCHS
warmup_steps = int(total_steps * train_cfg.get('warmup_ratio', 0.06))

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

USE_FP16 = train_cfg.get('fp16', False) and DEVICE.type == 'cuda'
scaler   = torch.cuda.amp.GradScaler() if USE_FP16 else None

early_stopping = EarlyStopping(
    patience=train_cfg['early_stopping_patience'],
    mode='max' if train_cfg['early_stopping_metric'] == 'macro_f1' else 'min',
)

CHECKPOINT_PATH = ARTIFACTS_DIR / out_cfg['checkpoint_name']

print(f'Total update steps : {total_steps:,}  (warmup: {warmup_steps})')
print(f'Effective batch    : {train_cfg["batch_size"] * ACCUM_STEPS}')
print(f'fp16               : {USE_FP16}')


## 6 — Training loop

In [ ]:
import time
from tqdm.auto import tqdm
from sklearn.metrics import f1_score


def _train_epoch(model, loader, optimizer, criterion, device,
                 grad_clip, accum_steps, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc='  train', leave=False), start=1):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        if scaler:
            with torch.autocast(device_type=device.type):
                logits = model(ids, mask)
                loss   = criterion(logits, lbls) / accum_steps
            scaler.scale(loss).backward()
        else:
            logits = model(ids, mask)
            loss   = criterion(logits, lbls) / accum_steps
            loss.backward()
        preds = logits.detach().argmax(dim=-1)
        correct    += (preds == lbls).sum().item()
        total      += lbls.size(0)
        total_loss += loss.item() * accum_steps * lbls.size(0)
        if step % accum_steps == 0 or step == len(loader):
            if scaler:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
    return total_loss / total, correct / total


@torch.no_grad()
def _eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    y_true, y_pred = [], []
    for batch in tqdm(loader, desc='  eval ', leave=False):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        logits = model(ids, mask)
        loss   = criterion(logits, lbls)
        total_loss += loss.item() * lbls.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
        y_true.extend(lbls.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return total_loss / total, correct / total, macro_f1, y_true, y_pred


print('Helper functions defined.')


In [ ]:
GRAD_CLIP   = train_cfg.get('gradient_clip', 1.0)
best_metric = float('-inf')
history     = []
t0          = time.time()

for epoch in range(1, EPOCHS + 1):
    t_ep = time.time()

    tr_loss, tr_acc = _train_epoch(
        model, train_loader, optimizer, criterion,
        DEVICE, GRAD_CLIP, ACCUM_STEPS, scaler,
    )
    val_loss, val_acc, val_f1, _, _ = _eval_epoch(
        model, val_loader, criterion, DEVICE,
    )

    monitor = val_f1 if train_cfg['early_stopping_metric'] == 'macro_f1' else -val_loss

    history.append({
        'epoch': epoch,
        'tr_loss': round(tr_loss, 4), 'tr_acc': round(tr_acc, 4),
        'val_loss': round(val_loss, 4), 'val_acc': round(val_acc, 4),
        'val_f1': round(val_f1, 4), 'time_s': round(time.time() - t_ep, 1),
    })

    print(
        f'Epoch {epoch:02d}/{EPOCHS}  '
        f'tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  '
        f'val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  '
        f'[{time.time()-t_ep:.0f}s]'
    )

    if monitor > best_metric:
        best_metric = monitor
        model.save_checkpoint(CHECKPOINT_PATH, epoch=epoch, best_metric=best_metric)
        print(f'  Checkpoint saved  (best val_f1={best_metric:.4f})')

    if early_stopping(monitor, epoch):
        print(f'\nEarly stopping at epoch {epoch}  (best epoch {early_stopping.best_epoch})')
        break

print(f'\nTraining done in {time.time()-t0:.1f}s  |  Best val_f1 = {best_metric:.4f}')
history_path = ARTIFACTS_DIR / out_cfg['history_name']
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'History saved -> {history_path}')


## 7 — Training curves

In [ ]:
import matplotlib.pyplot as plt

epochs_ax = [r['epoch']    for r in history]
tr_losses = [r['tr_loss']  for r in history]
va_losses = [r['val_loss'] for r in history]
tr_accs   = [r['tr_acc']   for r in history]
va_accs   = [r['val_acc']  for r in history]
va_f1s    = [r['val_f1']   for r in history]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_ax, tr_losses, label='train')
axes[0].plot(epochs_ax, va_losses, label='val')
axes[0].set_title('Cross-Entropy Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ax, tr_accs, label='train')
axes[1].plot(epochs_ax, va_accs, label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_ax, va_f1s, color='green', marker='o', label='val macro_f1')
best_ep = max(history, key=lambda r: r['val_f1'])
axes[2].axvline(best_ep['epoch'], linestyle='--', color='red', alpha=0.6,
                label=f'best epoch {best_ep["epoch"]} (f1={best_ep["val_f1"]:.4f})')
axes[2].set_title('Val Macro-F1'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = ARTIFACTS_DIR / 'bertweet_train_curves.png'
plt.savefig(curves_path, dpi=150)
plt.show()
print(f'Curves saved -> {curves_path}')


In [ ]:
df_hist = pd.DataFrame(history).set_index('epoch')
df_hist.style \
    .highlight_max(subset=['val_acc', 'val_f1'], color='#c6efce') \
    .highlight_min(subset=['tr_loss', 'val_loss'], color='#c6efce') \
    .format({c: '{:.4f}' for c in df_hist.columns if c != 'time_s'})


## 8 — Evaluation on test split

In [ ]:
from src.models.bertweet import BERTweetClassifier
from src.utils.metrics import compute_metrics, save_metrics, save_confusion_matrix_plot
from src.data.preprocess import ID_TO_LABEL

best_model = BERTweetClassifier.from_checkpoint(CHECKPOINT_PATH, device=DEVICE)
print(f'Loaded checkpoint: {CHECKPOINT_PATH}')

y_true, y_pred = [], []
best_model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc='test inference'):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['labels'].to(DEVICE)
        logits = best_model(ids, mask)
        preds  = logits.argmax(dim=-1)
        y_true.extend(lbls.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

metrics = compute_metrics(y_true, y_pred, label_names=ID_TO_LABEL,
                          model_name='bertweet', split='test')
metrics_path = ARTIFACTS_DIR / out_cfg['metrics_name']
save_metrics(metrics, metrics_path)

label_names = [ID_TO_LABEL.get(i, str(i)) for i in range(model_cfg['num_classes'])]
cm_path = ARTIFACTS_DIR / out_cfg['confusion_matrix_name']
save_confusion_matrix_plot(y_true, y_pred, label_names=label_names,
                           title='BERTweet - Test Confusion Matrix', save_path=cm_path)

print(f'\n{"="*45}')
print(f'  Accuracy    : {metrics["accuracy"]:.4f}')
print(f'  Macro  F1   : {metrics["macro_f1"]:.4f}')
print(f'  Weighted F1 : {metrics["weighted_f1"]:.4f}')
print(f'{"="*45}')
print(f'Metrics -> {metrics_path}')
print(f'Confusion matrix -> {cm_path}')


In [ ]:
per_class = metrics['per_class']
rows = []
for label_id, label_name in sorted(ID_TO_LABEL.items()):
    key = str(label_id)
    if key in per_class:
        rows.append({'label_id': label_id, 'label': label_name.title(), **per_class[key]})

df_pc = pd.DataFrame(rows).set_index('label_id')
df_pc.style \
    .background_gradient(subset=['f1-score'], cmap='RdYlGn') \
    .format({'precision': '{:.3f}', 'recall': '{:.3f}',
             'f1-score': '{:.3f}', 'support': '{:,.0f}'})


In [ ]:
from IPython.display import Image
Image(str(cm_path), width=700)
